In [2]:
import sys, os
sys.path.insert(1, '../..')

In [ ]:
import psycopg2
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from itertools import combinations
import config.config as conf

In [8]:
net_result = f'{conf.DATA_PATH}{conf.NETWORK_RESULT}'
net_src = f'{conf.DATA_PATH}{conf.NETWORK_SRC}'


In [12]:
conn = psycopg2.connect(host = conf.database_user['host'], dbname=conf.database_user['dbname'], user=conf.database_user['user'], password=conf.database_user['password'])
try:
    cur = conn.cursor()
    cur.execute("\
select \
                  x.q_id \
                , x.q_posttypeid \
                , x.q_acceptedanswerid \
                , x.q_parentid \
                , x.q_creationdate \
                , x.q_score \
                , x.q_viewcount \
                , x.q_owneruserid \
                , x.q_title \
                , x.q_tags \
                , x.q_answercount \
                , x.q_commentcount \
                , b.id               as a_id \
                , b.posttypeid       as a_posttypeid \
                , b.acceptedanswerid as a_acceptedanswerid \
                , b.parentid         as a_parentid \
                , b.creationdate     as a_creationdate \
                , b.score            as a_score \
                , b.viewcount        as a_viewcount \
                , b.owneruserid      as a_owneruserid \
                , b.title            as a_title \
                , b.tags             as a_tags \
                , b.answercount      as a_answercount \
                , b.commentcount     as a_commentcount \
  from ( \
           select a.id               as q_id \
                , a.posttypeid       as q_posttypeid \
                , a.acceptedanswerid as q_acceptedanswerid \
                , a.parentid         as q_parentid \
                , a.creationdate     as q_creationdate \
                , a.score            as q_score \
                , a.viewcount        as q_viewcount \
                , a.owneruserid      as q_owneruserid \
                , a.title            as q_title \
                , a.answercount      as q_answercount \
                , a.commentcount     as q_commentcount \
                , replace(replace(lower(a.tags), '<', ''), '>', ' ')as q_tags \
           from public_for_2324.posts a \
           where a.creationdate >= '2021-11-30' \
             and a.creationdate < '2024-12-01' \
             and a.posttypeid = '1' \
             and a.owneruserid is not null \
             and (replace(replace(lower(a.tags), '<', ''), '>', ' ') like '%python %') \
       )   x \
        , public_for_2324.posts b \
where b.parentid = x.q_id \
  and b.posttypeid = '2' \
  and b.owneruserid is not null \
; \
                " 
   )
    rows = cur.fetchall()
    

except psycopg2.DatabaseError as db_err:
    print(db_err)
finally : 
  cur.close()

In [13]:
df = pd.DataFrame(rows, columns = [
  'q_id' 
, 'q_posttypeid' 
, 'q_acceptedanswerid'
, 'q_parentid' 
, 'q_creationdate' 
, 'q_score' 
, 'q_viewcount' 
, 'q_owneruserid' 
, 'q_title' 
, 'q_tags' 
, 'q_answercount' 
, 'q_commentcount' 
, 'a_id' 
, 'a_posttypeid' 
, 'a_acceptedanswerid' 
, 'a_parentid' 
, 'a_creationdate' 
, 'a_score' 
, 'a_viewcount' 
, 'a_owneruserid' 
, 'a_title' 
, 'a_tags' 
, 'a_answercount'
, 'a_commentcount'
])

In [14]:
print(df['q_creationdate'].min())
print(df['a_creationdate'].max())

2021-11-30 00:00:11.787000
2025-06-30 19:45:59.467000


In [34]:
# 답변이 1년 이내의 달린것으로 조건 수정
df = df[df['a_creationdate'] - df['q_creationdate'] <= pd.Timedelta(days=365)]

In [38]:
df_2122 = df[(df['q_creationdate']>= '2021-11-30') & (df['q_creationdate']< '2022-11-30')  ]
df_2223 = df[(df['q_creationdate']>= '2022-11-30') & (df['q_creationdate']< '2023-11-30')  ]
df_2324 = df[(df['q_creationdate']>= '2023-11-30') & (df['q_creationdate']< '2024-11-30')  ]

In [42]:
def save_file(df, f_nm) :
    df_gephi = df[['q_owneruserid', 'a_owneruserid']]
    df_gephi.columns = ['Source', 'Target']
    print(f'{net_src}/{f_nm}.csv')
    df_gephi.to_csv(f'{net_src}/{f_nm}.csv')

In [43]:
save_file(df_2122, 'qna_2122')
save_file(df_2223, 'qna_2223')
save_file(df_2324, 'qna_2324')

/usr/share/d_ollama/data/network_src/qna_2122.csv
/usr/share/d_ollama/data/network_src/qna_2223.csv
/usr/share/d_ollama/data/network_src/qna_2324.csv
